# Competition-Grade Pipeline — Predict `BeatsPerMinute`

**Evaluation Metric:** Root Mean Squared Error (RMSE)

## Upgraded Pipeline
1. Data Loading & Preprocessing (ColumnTransformer, no aggressive feature selection)
2. 5-Fold Cross-Validation (leakage-safe)
3. Hyperparameter Tuning with Optuna (XGBoost + LightGBM)
4. Stacking Ensemble (XGB + LGBM + RF -> RidgeCV meta-learner)
5. Final Retrain on Full Data & Submission

---
## 0. Imports & Configuration

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV

# Gradient boosting
from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor

# Hyperparameter tuning
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Reproducibility
RANDOM_STATE = 42
N_FOLDS = 5
OPTUNA_FOLDS = 3       # fewer folds for speed during tuning
N_OPTUNA_TRIALS = 50
np.random.seed(RANDOM_STATE)

# Use sklearn's built-in scorer (avoids deprecated squared= param)
SCORER = 'neg_root_mean_squared_error'

print('All imports successful.')

All imports successful.


---
## 1. Data Loading & Preprocessing

In [2]:
# ---- Load data ----
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test  shape: {test_df.shape}')
print(f'Sample sub : {sample_sub.shape}')
train_df.head()

Train shape: (524164, 11)
Test  shape: (174722, 10)
Sample sub : (174722, 2)


,id,RhythmScore,AudioLoudness,VocalContent,AcousticQuality,InstrumentalScore,LivePerformanceLikelihood,MoodScore,TrackDurationMs,Energy,BeatsPerMinute
0,0,0.603610,-7.636942,0.023500,0.000005,0.000001,0.051385,0.409866,290715.6450,0.826267,147.53020
1,1,0.639451,-16.267598,0.071520,0.444929,0.349414,0.170522,0.651010,164519.5174,0.145400,136.15963
2,2,0.514538,-15.953575,0.110715,0.173699,0.453814,0.029576,0.423865,174495.5667,0.624667,55.31989
3,3,0.734463,-1.357000,0.052965,0.001651,0.159717,0.086366,0.278745,225567.4651,0.487467,147.91212
4,4,0.532968,-13.056437,0.023500,0.068687,0.000001,0.331345,0.477769,213960.6789,0.947333,89.58511


In [3]:
# ---- Separate target, IDs, features ----
TARGET = 'BeatsPerMinute'
ID_COL = 'id'

test_ids = test_df[ID_COL].copy()

y = train_df[TARGET].copy()
X = train_df.drop(columns=[ID_COL, TARGET])
X_test = test_df.drop(columns=[ID_COL])

print(f'X shape     : {X.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y shape     : {y.shape}')
print(f'\nTarget stats:\n{y.describe()}')

X shape     : (524164, 9)
X_test shape: (174722, 9)
y shape     : (524164,)

Target stats:
count    524164.000000
mean        119.034899
std          26.468077
min          46.718000
25%         101.070410
50%         118.747660
75%         136.686590
max         206.037000
Name: BeatsPerMinute, dtype: float64


In [4]:
# ---- Identify column types ----
numerical_cols   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numerical features   ({len(numerical_cols)}): {numerical_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')

# Check missing values
missing = X.isnull().sum()
print(f'\nMissing values:\n{missing[missing > 0] if missing.sum() > 0 else "None"}')

Numerical features   (9): ['RhythmScore', 'AudioLoudness', 'VocalContent', 'AcousticQuality', 'InstrumentalScore', 'LivePerformanceLikelihood', 'MoodScore', 'TrackDurationMs', 'Energy']
Categorical features (0): []

Missing values:
None


In [5]:
# ---- Build preprocessing ColumnTransformer ----
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

transformers = [('num', num_pipeline, numerical_cols)]
if categorical_cols:
    transformers.append(('cat', cat_pipeline, categorical_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

print('Preprocessor built successfully.')
print(f'  Numerical:   {len(numerical_cols)} features')
print(f'  Categorical: {len(categorical_cols)} features')

Preprocessor built successfully.
  Numerical:   9 features
  Categorical: 0 features


---
## 2. Preprocess & Validate Data Quality

In [6]:
# Pre-fit the preprocessor for Optuna speed
X_proc = preprocessor.fit_transform(X)

# Clean any NaN/Inf that StandardScaler may produce (e.g. zero-variance cols)
nan_count = np.isnan(X_proc).sum()
inf_count = np.isinf(X_proc).sum()
print(f'After preprocessing: shape={X_proc.shape}, NaNs={nan_count}, Infs={inf_count}')

if nan_count > 0 or inf_count > 0:
    X_proc = np.nan_to_num(X_proc, nan=0.0, posinf=0.0, neginf=0.0)
    print('Cleaned NaN/Inf values in preprocessed data.')

# Also check target
print(f'Target NaNs: {y.isna().sum()}')

After preprocessing: shape=(524164, 9), NaNs=0, Infs=0
Target NaNs: 0


In [7]:
# Quick sanity check: train a small XGBoost to confirm everything works
sanity_model = XGBRegressor(n_estimators=50, max_depth=4, random_state=RANDOM_STATE,
                            n_jobs=-1, verbosity=0)
sanity_kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
sanity_scores = cross_val_score(sanity_model, X_proc, y, cv=sanity_kf,
                                scoring=SCORER, n_jobs=1)
print(f'Sanity check RMSE: {-sanity_scores.mean():.5f} (should be a real number, not NaN)')
assert not np.isnan(sanity_scores).any(), 'ERROR: Scores contain NaN!'
print('Sanity check PASSED -- ready for Optuna.')

Sanity check RMSE: 26.47496 (should be a real number, not NaN)
Sanity check PASSED -- ready for Optuna.


---
## 3. Hyperparameter Tuning with Optuna

**Speed optimizations:**
- 3-Fold CV during tuning (5-Fold for final eval)
- `n_jobs=1` on `cross_val_score` to avoid Windows nested-parallelism deadlocks
- Models use `n_jobs=-1` internally for tree-level parallelism

In [8]:
# KFold for Optuna (3-fold for speed)
optuna_kf = KFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f'Optuna config: {N_OPTUNA_TRIALS} trials, {OPTUNA_FOLDS}-fold CV')
print(f'Estimated time per trial: ~1-2 min')
print(f'Estimated total: ~{N_OPTUNA_TRIALS * 1.5:.0f} min per model')

Optuna config: 50 trials, 3-fold CV
Estimated time per trial: ~1-2 min
Estimated total: ~75 min per model


In [9]:
# ---- Optuna objective for XGBoost ----
def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'random_state':     RANDOM_STATE,
        'n_jobs':           -1,
        'verbosity':        0
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X_proc, y, cv=optuna_kf,
                             scoring=SCORER, n_jobs=1)
    return -scores.mean()  # return positive RMSE to minimize

print(f'Starting XGBoost Optuna study ({N_OPTUNA_TRIALS} trials)...')
xgb_study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name='xgb_tuning'
)
xgb_study.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\nCompleted trials: {len([t for t in xgb_study.trials if t.state.name == "COMPLETE"])}/{N_OPTUNA_TRIALS}')
print(f'XGBoost best CV RMSE: {xgb_study.best_value:.5f}')
print(f'Best params: {xgb_study.best_params}')

Starting XGBoost Optuna study (50 trials)...


Best trial: 44. Best value: 26.4606: 100%|██████████| 50/50 [32:53<00:00, 39.47s/it]


Completed trials: 50/50
XGBoost best CV RMSE: 26.46056
Best params: {'n_estimators': 358, 'learning_rate': 0.012583512389964413, 'max_depth': 3, 'subsample': 0.6831823583659804, 'colsample_bytree': 0.9120874881346046, 'reg_alpha': 0.011918981812918967, 'reg_lambda': 1.5184404420560319e-07, 'min_child_weight': 10}


In [11]:
# ---- Optuna objective for LightGBM ----
def lgbm_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 150),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
        'random_state':     RANDOM_STATE,
        'n_jobs':           -1,
        'verbosity':        -1
    }
    model = LGBMRegressor(**params)
    scores = cross_val_score(model, X_proc, y, cv=optuna_kf,
                             scoring=SCORER, n_jobs=1)
    return -scores.mean()

print(f'Starting LightGBM Optuna study ({N_OPTUNA_TRIALS} trials)...')
lgbm_study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name='lgbm_tuning'
)
lgbm_study.optimize(lgbm_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\nCompleted trials: {len([t for t in lgbm_study.trials if t.state.name == "COMPLETE"])}/{N_OPTUNA_TRIALS}')
print(f'LightGBM best CV RMSE: {lgbm_study.best_value:.5f}')
print(f'Best params: {lgbm_study.best_params}')

Starting LightGBM Optuna study (50 trials)...


Best trial: 28. Best value: 26.4607: 100%|██████████| 50/50 [17:58<00:00, 21.58s/it]


Completed trials: 50/50
LightGBM best CV RMSE: 26.46071
Best params: {'n_estimators': 309, 'learning_rate': 0.014364189010970025, 'max_depth': 4, 'num_leaves': 146, 'subsample': 0.942074278890653, 'colsample_bytree': 0.6269381819283107, 'reg_alpha': 4.078867594284913e-05, 'reg_lambda': 2.1315935605831153e-07, 'min_child_samples': 21}


In [13]:
# ---- Build tuned models from Optuna results ----
best_xgb = XGBRegressor(
    **xgb_study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)

best_lgbm = LGBMRegressor(
    **lgbm_study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

# Random Forest with reasonable defaults (diversity base learner)
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print('Tuned models instantiated:')
print(f'  XGBoost:  {xgb_study.best_value:.5f} (CV RMSE)')
print(f'  LightGBM: {lgbm_study.best_value:.5f} (CV RMSE)')
print(f'  RF:       default hyperparams (diversity base learner)')

Tuned models instantiated:
  XGBoost:  26.46056 (CV RMSE)
  LightGBM: 26.46071 (CV RMSE)
  RF:       default hyperparams (diversity base learner)


---
## 4. Model Ensembling -- StackingRegressor

Base estimators: tuned XGBoost, tuned LightGBM, Random Forest  
Meta-learner: RidgeCV (automatically selects regularization strength)

In [14]:
# ---- Build the StackingRegressor ----
stacking_model = StackingRegressor(
    estimators=[
        ('xgb',  best_xgb),
        ('lgbm', best_lgbm),
        ('rf',   rf_model)
    ],
    final_estimator=RidgeCV(alphas=np.logspace(-4, 4, 20)),
    cv=KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1,  # avoid nested parallelism on Windows
    passthrough=False
)

print('StackingRegressor built.')
print('  Base learners: XGBoost (tuned), LightGBM (tuned), RandomForest')
print('  Meta-learner:  RidgeCV')

StackingRegressor built.
  Base learners: XGBoost (tuned), LightGBM (tuned), RandomForest
  Meta-learner:  RidgeCV


In [15]:
# ---- Evaluate stacking with 5-Fold CV ----
full_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('stack', stacking_model)
])

eval_kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f'Evaluating StackingRegressor with {N_FOLDS}-Fold CV...')
print('(This will take a while -- stacking does internal CV per outer fold)\n')

stacking_cv_scores = cross_val_score(
    full_pipeline, X, y,
    cv=eval_kf, scoring=SCORER, n_jobs=1
)
stacking_rmse = -stacking_cv_scores

print('=' * 55)
print('  STACKING REGRESSOR -- 5-Fold CV Results')
print('=' * 55)
for i, score in enumerate(stacking_rmse, 1):
    print(f'  Fold {i}: RMSE = {score:.5f}')
print('-' * 55)
print(f'  Mean RMSE: {stacking_rmse.mean():.5f} (+/- {stacking_rmse.std():.5f})')
print('=' * 55)

Evaluating StackingRegressor with 5-Fold CV...
(This will take a while -- stacking does internal CV per outer fold)

  STACKING REGRESSOR -- 5-Fold CV Results
  Fold 1: RMSE = 26.43877
  Fold 2: RMSE = 26.48410
  Fold 3: RMSE = 26.52369
  Fold 4: RMSE = 26.44367
  Fold 5: RMSE = 26.40798
-------------------------------------------------------
  Mean RMSE: 26.45964 (+/- 0.04015)


In [16]:
# ---- Individual model comparison ----
def cv_rmse_simple(model, X_data, y_data):
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_data, y_data, cv=eval_kf,
                             scoring=SCORER, n_jobs=1)
    rmse_scores = -scores
    return rmse_scores.mean(), rmse_scores.std()

print('Individual model CV RMSE for comparison:\n')

xgb_mean, xgb_std = cv_rmse_simple(best_xgb, X, y)
print(f'  XGBoost (tuned):  {xgb_mean:.5f} +/- {xgb_std:.5f}')

lgbm_mean, lgbm_std = cv_rmse_simple(best_lgbm, X, y)
print(f'  LightGBM (tuned): {lgbm_mean:.5f} +/- {lgbm_std:.5f}')

rf_mean, rf_std = cv_rmse_simple(rf_model, X, y)
print(f'  RandomForest:     {rf_mean:.5f} +/- {rf_std:.5f}')

print(f'\n  Stacking:         {stacking_rmse.mean():.5f} +/- {stacking_rmse.std():.5f}')

Individual model CV RMSE for comparison:

  XGBoost (tuned):  26.45971 +/- 0.04007
  LightGBM (tuned): 26.46004 +/- 0.04030
  RandomForest:     26.58629 +/- 0.04420

  Stacking:         26.45964 +/- 0.04015


---
## 5. Final Training & Submission

In [17]:
# ---- Retrain on full training data ----
print('Retraining full pipeline on entire train.csv...')
full_pipeline.fit(X, y)
print(f'Training complete ({X.shape[0]} samples).')

# ---- Generate test predictions ----
test_predictions = full_pipeline.predict(X_test)
print(f'Test predictions: {test_predictions.shape}')
print(f'Prediction range: [{test_predictions.min():.2f}, {test_predictions.max():.2f}]')

Retraining full pipeline on entire train.csv...
Training complete (524164 samples).
Test predictions: (174722,)
Prediction range: [115.58, 125.69]


In [18]:
# ---- Create and export submission ----
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: test_predictions
})

# Sanity checks
assert submission.shape[0] == sample_sub.shape[0], \
    f'Row count mismatch: {submission.shape[0]} vs {sample_sub.shape[0]}'
assert list(submission.columns) == list(sample_sub.columns), \
    f'Column mismatch: {list(submission.columns)} vs {list(sample_sub.columns)}'

submission.to_csv('submission_stacked.csv', index=False)
print('submission_stacked.csv saved!')
print(f'Shape: {submission.shape}')
submission.head(10)

submission_stacked.csv saved!
Shape: (174722, 2)


,id,BeatsPerMinute
0,524164,119.334547
1,524165,118.557341
2,524166,119.483691
3,524167,119.268223
4,524168,119.389686
5,524169,119.072836
6,524170,118.636769
7,524171,118.547370
8,524172,118.979797
9,524173,118.443715


In [19]:
print('\n' + '=' * 55)
print('  PIPELINE COMPLETE')
print('=' * 55)
print(f'  Stacking CV RMSE : {stacking_rmse.mean():.5f}')
print(f'  XGBoost CV RMSE  : {xgb_mean:.5f}')
print(f'  LightGBM CV RMSE : {lgbm_mean:.5f}')
print(f'  RF CV RMSE       : {rf_mean:.5f}')
print(f'  Submission rows  : {submission.shape[0]}')
print(f'  Output file      : submission_stacked.csv')
print('=' * 55)


  PIPELINE COMPLETE
  Stacking CV RMSE : 26.45964
  XGBoost CV RMSE  : 26.45971
  LightGBM CV RMSE : 26.46004
  RF CV RMSE       : 26.58629
  Submission rows  : 174722
  Output file      : submission_stacked.csv
